# Social Post Yield Pipeline (Model 5) — CRISP-DM

Follows **CRISP-DM** (six phases), aligned with the course ML pipeline template. **Regression** target: `donation_referrals`.

## Phase 1: Business Understanding
**Objective:** Predict **donation referrals** from post attributes for campaign planning.
**Success metrics:** MAE, RMSE, R² vs a **mean-value baseline** on a chronological holdout.

## Phase 2: Data Understanding
**Source:** `social_media_posts.csv` (see `ml-pipelines/data/README.md`). EDA in code after load.

## Phase 3: Data Preparation
Excludes post-outcome leakage features; chronological split; preprocessing in sklearn `Pipeline`.

## Phase 4: Modeling
Random Forest regressor + Linear Regression (interpretable).

## Phase 5: Evaluation
Compare to **DummyRegressor(strategy='mean')**; permutation importance.

## Phase 6: Deployment
**joblib** below. Production: `model_5_train.py` / `model_5_score.py`, API `/api/ml/model5/*`, UI social insights pages.


### Feature scope (leakage-safe)

**Goal:** Predict `donation_referrals` from attributes available **before/at** post time.

- Excludes outcome metrics (e.g. impressions, reach, likes, engagement_rate).
- Chronological train/test split.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import joblib


def resolve_csv(name: str) -> Path:
    cwd = Path.cwd()
    for base in (cwd, cwd / 'ml-pipelines', cwd / 'data', cwd / 'ml-pipelines' / 'data'):
        q = base / name
        if q.exists():
            return q
    raise FileNotFoundError(
        f'Could not find {name}. Place it in ., data/, ml-pipelines/, or ml-pipelines/data/ (see ml-pipelines/data/README.md).'
    )


DATA_DIR = resolve_csv('social_media_posts.csv').parent
ARTIFACTS_DIR = (Path('ml-pipelines') / 'artifacts') if (Path.cwd() / 'ml-pipelines').is_dir() else (DATA_DIR / 'artifacts')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print('Using DATA_DIR:', DATA_DIR.resolve())
print('Writing artifacts to:', ARTIFACTS_DIR.resolve())
posts = pd.read_csv(DATA_DIR / 'social_media_posts.csv', parse_dates=['created_at'])

print('=== Phase 2: EDA ===')
print(posts.shape)
print(posts[['donation_referrals']].describe() if 'donation_referrals' in posts.columns else posts.describe(include='all').T.head(15))

def time_split(df, time_col, frac=0.8):
    df = df.sort_values(time_col).copy()
    cut = int(len(df) * frac)
    return df.iloc[:cut].copy(), df.iloc[cut:].copy()

def build_preprocessor(X):
    num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat_cols = [c for c in X.columns if c not in num_cols]
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('pow', PowerTransformer(method='yeo-johnson', standardize=False)), ('sc', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
    ])



In [ ]:
from sklearn.dummy import DummyRegressor

feature_cols = [
    'platform', 'day_of_week', 'post_hour', 'post_type', 'media_type', 'num_hashtags',
    'mentions_count', 'has_call_to_action', 'call_to_action_type', 'content_topic',
    'sentiment_tone', 'caption_length', 'features_resident_story', 'campaign_name',
    'is_boosted', 'boost_budget_php'
]
cols = [c for c in feature_cols + ['donation_referrals', 'created_at'] if c in posts.columns]
m5 = posts[cols].copy()

train_df, test_df = time_split(m5, 'created_at', 0.8)
X_train = train_df.drop(columns=['donation_referrals', 'created_at'])
y_train = train_df['donation_referrals']
X_test = test_df.drop(columns=['donation_referrals', 'created_at'])
y_test = test_df['donation_referrals']

pre = build_preprocessor(X_train)
predictive = Pipeline([('pre', pre), ('model', RandomForestRegressor(n_estimators=350, min_samples_leaf=4, random_state=42))])
explanatory = Pipeline([('pre', pre), ('model', LinearRegression())])

predictive.fit(X_train, y_train)
explanatory.fit(X_train, y_train)

pred = predictive.predict(X_test)
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)
d_pred = dummy.predict(X_test)
print('=== Phase 5: Baseline mean DummyRegressor (test) ===')
print({'mae': mean_absolute_error(y_test, d_pred), 'rmse': mean_squared_error(y_test, d_pred) ** 0.5, 'r2': r2_score(y_test, d_pred)})
print('=== Phase 5: Random Forest regressor (test) ===')
mse = mean_squared_error(y_test, pred)
rmse = mse ** 0.5
print({'mae': mean_absolute_error(y_test, pred), 'rmse': rmse, 'r2': r2_score(y_test, pred)})  # RF

imp = permutation_importance(predictive, X_test, y_test, n_repeats=8, random_state=42)
print(pd.DataFrame({'feature': X_test.columns, 'importance': imp.importances_mean}).sort_values('importance', ascending=False).head(10))

joblib.dump(predictive, ARTIFACTS_DIR / 'model5_predictive.joblib')
joblib.dump(explanatory, ARTIFACTS_DIR / 'model5_explanatory.joblib')



In [ ]:
# Final business insights block (human-readable + actionable)

print('\n=== BUSINESS TAKEAWAYS: MODEL 5 (POST DONATION YIELD) ===')

# Build prediction table on all posts for strategy insights
scored = m5.copy()
feature_inputs = [c for c in scored.columns if c not in ['donation_referrals', 'created_at']]
scored['pred_referrals'] = predictive.predict(scored[feature_inputs])

baseline_pred = float(scored['pred_referrals'].mean())
print(f'Baseline expected referrals/post (model): {baseline_pred:.2f}')

# 1) Best 5 posting windows (day + hour)
win_tbl = (
    scored.groupby(['day_of_week', 'post_hour'], dropna=False)['pred_referrals']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .head(5)
)
win_tbl['expected_uplift_vs_baseline_pct'] = ((win_tbl['pred_referrals'] / baseline_pred) - 1.0) * 100
print('\nBest 5 posting windows:')
print(win_tbl.to_string(index=False))

# 2) Best post types by platform
ptype_tbl = (
    scored.groupby(['platform', 'post_type'], dropna=False)['pred_referrals']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
best_ptype_per_platform = ptype_tbl.groupby('platform', as_index=False).head(1).copy()
best_ptype_per_platform['expected_uplift_vs_baseline_pct'] = ((best_ptype_per_platform['pred_referrals'] / baseline_pred) - 1.0) * 100
print('\nBest post type by platform:')
print(best_ptype_per_platform.sort_values('pred_referrals', ascending=False).to_string(index=False))

# 3) Story vs no-story uplift
if 'features_resident_story' in scored.columns:
    story_tbl = scored.groupby('features_resident_story', dropna=False)['pred_referrals'].mean().reset_index()
    print('\nResident story effect (predicted):')
    print(story_tbl.to_string(index=False))

# 4) Executive-readable guidance
top_window = win_tbl.iloc[0]
print('\nActionable guidance:')
print(f"- Prioritize posting around {top_window['day_of_week']} at hour {int(top_window['post_hour']) if pd.notna(top_window['post_hour']) else 'N/A'}.")
print('- Use the best post type per platform table to choose format before publishing.')
print('- Compare expected uplift vs baseline to prioritize high-leverage content in weekly planning.')
print('- Treat these as predictive recommendations (ranking), not causal proof.')
